<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/04_introduction_to_altair.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Data Visualization with Altair


## 1: Setup and Imports

Ensure necessary libraries are installed. In Google Colab, some common libraries are pre-installed. If running locally or in a different environment, you might need to run:

`!pip install altair vega_datasets polars`


In [ ]:

import altair as alt
import polars as pl
from vega_datasets import data as vega_data # For loading sample datasets

# Enable the Altair renderer for Google Colab
# If you're using a different environment (like JupyterLab or a classic Notebook),
# you might need to change this. Examples:
# alt.renderers.enable('jupyterlab')
# alt.renderers.enable('notebook')
# alt.renderers.enable('default')
alt.renderers.enable('colab')

print("Libraries imported and Altair renderer enabled for Colab.")


Libraries imported and Altair renderer enabled for Colab.



### 1.1 Why Visualize Data? Anscombe's Quartet

Summary statistics can be informative, but they don't tell the whole story.

Anscombe's Quartet demonstrates this perfectly.


In [ ]:
# Load Anscombe's Quartet from vega_datasets
anscombe_pd_df = vega_data.anscombe() # This loads as a Pandas DataFrame

# Convert to a Polars DataFrame (as we're using Polars in this course)
anscombe_pl_df = pl.from_pandas(anscombe_pd_df)

# Let's inspect the data structure
print("First few rows of Anscombe's Quartet (Polars DataFrame):")
print(anscombe_pl_df.head())

First few rows of Anscombe's Quartet (Polars DataFrame):
shape: (5, 3)
┌────────┬─────┬──────┐
│ Series ┆ X   ┆ Y    │
│ ---    ┆ --- ┆ ---  │
│ str    ┆ i64 ┆ f64  │
╞════════╪═════╪══════╡
│ I      ┆ 10  ┆ 8.04 │
│ I      ┆ 8   ┆ 6.95 │
│ I      ┆ 13  ┆ 7.58 │
│ I      ┆ 9   ┆ 8.81 │
│ I      ┆ 11  ┆ 8.33 │
└────────┴─────┴──────┘


In [ ]:

# Now, let's calculate key summary statistics for each dataset within the quartet.
# We'll group by the 'Dataset' column.

summary_stats = anscombe_pl_df.group_by("Series").agg(
    pl.mean("X").alias("Mean_X"),
    pl.std("X").alias("StdDev_X"),
    pl.mean("Y").alias("Mean_Y"),
    pl.std("Y").alias("StdDev_Y"),
    pl.corr("X", "Y").alias("Correlation_XY")
).sort("Series") # Sort for consistent display

print("\nSummary Statistics for Anscombe's Quartet:")
print(summary_stats)



Summary Statistics for Anscombe's Quartet:
shape: (4, 6)
┌────────┬────────┬──────────┬──────────┬──────────┬────────────────┐
│ Series ┆ Mean_X ┆ StdDev_X ┆ Mean_Y   ┆ StdDev_Y ┆ Correlation_XY │
│ ---    ┆ ---    ┆ ---      ┆ ---      ┆ ---      ┆ ---            │
│ str    ┆ f64    ┆ f64      ┆ f64      ┆ f64      ┆ f64            │
╞════════╪════════╪══════════╪══════════╪══════════╪════════════════╡
│ I      ┆ 9.0    ┆ 3.316625 ┆ 7.5      ┆ 2.03289  ┆ 0.816186       │
│ II     ┆ 9.0    ┆ 3.316625 ┆ 7.500909 ┆ 2.031657 ┆ 0.816237       │
│ III    ┆ 9.0    ┆ 3.316625 ┆ 7.5      ┆ 2.030424 ┆ 0.816287       │
│ IV     ┆ 9.0    ┆ 3.316625 ┆ 7.500909 ┆ 2.030579 ┆ 0.816521       │
└────────┴────────┴──────────┴──────────┴──────────┴────────────────┘


**Note: The summary statistics (mean, std dev, correlation) are nearly identical for all four datasets!**

### 1.2 Visualizing Anscombe's Quartet with Altair

 Now, let's see what these datasets *look* like.
 We will create a scatter plot for each dataset.

 **The core Altair syntax: `alt.Chart(data).mark_type().encode(visual_channels)`**

 We specify the data type for X and Y as Quantitative ('Q') using a colon.

 This helps Altair apply appropriate scales and axes.
 Example: `x='X:Q'`


In [ ]:
def dataset_mapper(val: str) -> str:
  dataset_map = {
    'I': 'Linear',
    'II': 'Non-linear',
    'III': 'Linear with outlier',
    'IV': 'Vertical line with outlier'
  }
  return dataset_map.get(val, 'Unknown')

anscombe = anscombe_pl_df.select(
    pl.col('X', 'Y'),
    pl.col('Series')
      .map_elements(dataset_mapper, return_dtype=pl.String)
)

(
    alt.Chart(anscombe)
    .mark_point(size=60)
    .encode(
        alt.X('X:Q', scale=alt.Scale(domain=[0, 20])),
        alt.Y('Y:Q', scale=alt.Scale(domain=[0, 14])),
        alt.Color('Series:N')
          .legend(title="Dataset"),
        alt.Facet('Series:N')
        .columns(2)
        .title(None)
    )
    .properties(
        title="Anscombe's Quartet",
        width=200,
        height=200,
    )
)

alt.Chart(...)

How does the visualization compare with the summary statistics?
- Dataset I: Appears to be a linear relationship.
- Dataset II: Shows a clear non-linear (curved) relationship.
- Dataset III: A linear relationship with a significant outlier.
- Dataset IV: Most X values are constant, with one influential outlier.

Visualization helps identify patterns, anomalies, and guides further analysis.



### 1.3 Understanding Altair's Declarative Nature & Core Idea

Recall: `object = data + behavior`

You've just used this syntax to create the Anscombe plots!
- `alt.Chart(anscombe)`: Create a `Chart` object with `anscombe` **data**.
- `.mark_point()`: Ask the chart object to use  `point` as **mark** type.
- `.encode(x='X:Q', y='Y:Q')`: Ask chart to **encode** `X` and `Y` columns
  as **Q**uantitative  data type to visual properties (channels) `x-position` and `y-position`.

Any diagram you create in Altair follow this basic structure. This consistent structure is key to Altair's power and ease of use.
By focusing on what you want to represent, you can build a wide variety of charts.

In [26]:
from vega_datasets import data as vega_data

# Load as Pandas DataFrame first
cars_pd_df = vega_data.cars()

# Convert to Polars DataFrame
cars_pl_df = pl.from_pandas(cars_pd_df)


(406, 9)
shape: (1, 9)
┌──────┬──────────────┬───────────┬──────────────┬───┬──────────────┬──────────────┬──────┬────────┐
│ Name ┆ Miles_per_Ga ┆ Cylinders ┆ Displacement ┆ … ┆ Weight_in_lb ┆ Acceleration ┆ Year ┆ Origin │
│ ---  ┆ llon         ┆ ---       ┆ ---          ┆   ┆ s            ┆ ---          ┆ ---  ┆ ---    │
│ u32  ┆ ---          ┆ u32       ┆ u32          ┆   ┆ ---          ┆ u32          ┆ u32  ┆ u32    │
│      ┆ u32          ┆           ┆              ┆   ┆ u32          ┆              ┆      ┆        │
╞══════╪══════════════╪═══════════╪══════════════╪═══╪══════════════╪══════════════╪══════╪════════╡
│ 0    ┆ 8            ┆ 0         ┆ 0            ┆ … ┆ 0            ┆ 0            ┆ 0    ┆ 0      │
└──────┴──────────────┴───────────┴──────────────┴───┴──────────────┴──────────────┴──────┴────────┘


In [31]:
# Always a good idea to inspect your data
cars_pl_df.head()
cars_pl_df.shape
cars_pl_df.null_count() # Check for missing values

Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
str,f64,i64,f64,f64,i64,f64,datetime[ns],str
"""chevrolet chevelle malibu""",18.0,8,307.0,130.0,3504,12.0,1970-01-01 00:00:00,"""USA"""
"""buick skylark 320""",15.0,8,350.0,165.0,3693,11.5,1970-01-01 00:00:00,"""USA"""
"""plymouth satellite""",18.0,8,318.0,150.0,3436,11.0,1970-01-01 00:00:00,"""USA"""
"""amc rebel sst""",16.0,8,304.0,150.0,3433,12.0,1970-01-01 00:00:00,"""USA"""
"""ford torino""",17.0,8,302.0,140.0,3449,10.5,1970-01-01 00:00:00,"""USA"""


(406, 9)

Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,8,0,0,6,0,0,0,0



## 2: Your First Plots with Altair

### 2.0 Load and Prepare the Dataset for Visualization

In [56]:
# Load the 'cars' dataset from vega_datasets
cars_pd = vega_data.cars()

# Convert to a Polars DataFrame
cars_pl = pl.from_pandas(cars_pd)

print("Cars dataset loaded into a Polars DataFrame:")

cars_pl.head()

print(f"\nShape of the cars dataset: {cars_pl.shape}")

# Quick check for missing values (important before plotting some fields)
print("\nNull counts per column:")
cars_pl.null_count()


# For some plots, it's useful to drop rows with missing values in key columns.
# For example, if plotting Horsepower vs. Miles_per_Gallon:
cars_pl_cleaned = cars_pl.drop_nulls(subset=['Horsepower', 'Miles_per_Gallon', 'Origin', 'Name'])

print(f"\nShape after dropping some nulls: {cars_pl_cleaned.shape}")

Cars dataset loaded into a Polars DataFrame:


Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
str,f64,i64,f64,f64,i64,f64,datetime[ns],str
"""chevrolet chevelle malibu""",18.0,8,307.0,130.0,3504,12.0,1970-01-01 00:00:00,"""USA"""
"""buick skylark 320""",15.0,8,350.0,165.0,3693,11.5,1970-01-01 00:00:00,"""USA"""
"""plymouth satellite""",18.0,8,318.0,150.0,3436,11.0,1970-01-01 00:00:00,"""USA"""
"""amc rebel sst""",16.0,8,304.0,150.0,3433,12.0,1970-01-01 00:00:00,"""USA"""
"""ford torino""",17.0,8,302.0,140.0,3449,10.5,1970-01-01 00:00:00,"""USA"""



Shape of the cars dataset: (406, 9)

Null counts per column:


Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,8,0,0,6,0,0,0,0



Shape after dropping some nulls: (392, 9)


### 2.1 Scatter Plots: Exploring Relationships

- **Scatter plots** help visualize the relationship between two quantitative variables.
- **Mark**: `mark_point()` or `mark_circle()`
- **Encodings**:
  - `x`,
  - `y`,
  - `color`,
  - `size`,
  - `tooltip`



#### **Student Task**:
1. Modify the scatter plot below to show 'Acceleration:Q' on the x-axis and 'Weight_in_lbs:Q' on the y-axis.
2. Keep the color encoding by 'Origin'.
3. Update the tooltips and title appropriately.
4. Comment out the code that is not relevant for `Acceleration` and `Weight_in_lbs`

In [149]:
# Data prep
mpg_col = pl.col('Miles_per_Gallon')
MPG_summary = cars_pl_cleaned.select(
    mpg_col.mean().alias('Mean_MPG'),
    mpg_col.median().alias('Median_MPG'),
    mpg_col.min().alias('Min_MPG'),
    mpg_col.max().alias('Max_MPG'),
    mpg_col.lt(mpg_col.median()).sum().alias('Below_Median'),
    mpg_col.lt(mpg_col.mean()).sum().alias('Below_Mean')
).to_dicts()[0]

# Incremental chart build

base = alt.Chart(cars_pl_cleaned)

points = base.mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N',
    tooltip=['Name:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
)

median_rule = base.mark_rule(color="light-pink", strokeDash=[2,2], opacity=0.5).encode(
    y = alt.datum(MPG_summary['Median_MPG']),
)

max_rule = base.mark_rule(color="green", strokeDash=[2,2], opacity=0.5).encode(
    y = alt.datum(MPG_summary['Max_MPG']),
)

median_text = base.mark_text(
      dx=650,
      dy=5,
      fontSize=12,
      fontWeight='normal'
    ).encode(
    y = alt.datum(MPG_summary['Median_MPG']),
    x = alt.datum(0),
    text = alt.value('Median MPG'),

)

# Chart display
(points + median_rule + max_rule + median_text).properties(
    title='Car Horsepower vs. Miles per Gallon by Origin',
    width=600,
    height=400
)

alt.LayerChart(...)

In [ ]:
# Your code here

### 2.2 Bar Charts: Comparing Categories or Showing Counts

- **Bar charts** compare a quantitative measure across different categories.
- **Mark**: mark_bar()
- **Encodings**:
  - `x` (categorical),
  - `y` (quantitative/aggregate),
  - `color`

In [154]:
# Example 1: Number of cars from each 'Origin' (Frequency)
(
    alt.Chart(cars_pl_cleaned)
    .mark_bar()
    .encode(
      x='Origin:N',
      y='count():Q', # Altair's way to count occurrences
      color='Origin:N', # Optional: color bars by origin
      tooltip=['Origin:N', 'count():Q']
    )
    .properties(
      title='Number of Cars by Origin',
      width=300,
      height=300
    )
)

alt.Chart(...)

In [156]:
# Example 2: Average 'Horsepower' for each 'Origin'.
# Altair can do simple aggregations like 'average', 'sum', 'min', 'max'.
alt.Chart(cars_pl_cleaned).mark_bar().encode(
    x='Origin:N',
    y='average(Horsepower):Q',
    color='Origin:N',
    tooltip=['Origin:N', 'average(Horsepower):Q']
).properties(
    title='Average Horsepower by Origin',
    height=300,
    width=300
)

alt.Chart(...)

In [158]:
# Alternatively, pre-aggregate with Polars for more complex scenarios or clarity:
avg_hp_origin_polars = cars_pl_cleaned.group_by('Origin').agg(
    pl.mean('Horsepower').alias('Mean_Horsepower')
).sort('Origin')

alt.Chart(avg_hp_origin_polars).mark_bar().encode(
    x='Origin:N',
    y='Mean_Horsepower:Q',
    color='Origin:N', # Or a fixed color: alt.value('steelblue')
    tooltip=['Origin:N', 'Mean_Horsepower:Q']
).properties(
    title='Average Horsepower by Origin (Polars Pre-aggregated)',
    height=300,
    width=300
)

alt.Chart(...)



#### Student Task:

Create a bar chart showing the average 'Displacement' for cars with different numbers of 'Cylinders'.
  - X-axis: 'Cylinders:O' (Ordinal, as cylinder count has an order)
  - Y-axis: Average 'Displacement'
  - Pre-aggregate the data using Polars.
  - Add appropriate tooltips and a title.


In [ ]:
# YOUR CODE HERE for Student Task (Bar Chart)


### 2.3 Line Charts: Showing Trends Over Time or Sequence

- Line charts are excellent for visualizing trends.
- Mark: mark_line()
- Encodings:
  - `x` (temporal or ordered)
  - `y` (quantitative)
  - `color` (for multiple series)


In [168]:
# Load the 'seattle-weather' dataset
weather_pd = vega_data.seattle_weather()
weather_pl = pl.from_pandas(weather_pd)
weather_pl.head()

date,precipitation,temp_max,temp_min,wind,weather
datetime[ns],f64,f64,f64,f64,str
2012-01-01 00:00:00,0.0,12.8,5.0,4.7,"""drizzle"""
2012-01-02 00:00:00,10.9,10.6,2.8,4.5,"""rain"""
2012-01-03 00:00:00,0.8,11.7,7.2,2.3,"""rain"""
2012-01-04 00:00:00,20.3,12.2,5.6,4.7,"""rain"""
2012-01-05 00:00:00,1.3,8.9,2.8,6.1,"""rain"""


In [166]:

alt.Chart(weather_pl).mark_line().encode(
    x='date:T',
    y='temp_max:Q',
    color='weather:N', # Different line for each weather type
    tooltip=['date:T', 'temp_max:Q', 'weather:N']
).properties(
    title='Maximum Daily Temperature in Seattle by Weather Condition',
    width=600,
    height=350
)

alt.Chart(...)

#### Student Task:

Using the `seattle_weather` dataset:
1. Create a line chart showing the 'precipitation' over 'date'.
2. Only include data for days where precipitation was greater than 0. (Hint: Filter with Polars first).
3. Do NOT color by weather type for this one (i.e., a single line).
4. Add appropriate tooltips and a title.


In [169]:
# YOUR CODE HERE

### 2.4 Histograms: Visualizing Distributions

- **Histograms** show the distribution of a single quantitative variable.
- `Mark`: mark_bar()
- `Encodings`:
  - `x` (quantitative, binned)
  - `y` (count)


In [159]:
# Use `alt.X()` for more control over binning.
alt.Chart(cars_pl_cleaned).mark_bar().encode(
    alt.X(
        'Miles_per_Gallon:Q',
        bin=alt.Bin(maxbins=15),
        title='Miles per Gallon'), # Explicitly define bins
    y='count():Q',
    tooltip=[alt.Tooltip('Miles_per_Gallon:Q', bin=True), 'count():Q'] # Show binned range in tooltip
).properties(
    title='Distribution of Miles per Gallon',
    width=400
)


alt.Chart(...)

#### Student Task:
 1. Create a histogram for the 'Horsepower' column from the `cars_pl_cleaned` DataFrame.
 2. Experiment with the `maxbins` parameter (e.g., 10, 20, 30) to see how it changes the plot.
 3. Add appropriate tooltips and a title.

alt.Chart(...)

In [ ]:
# Your code goes here

### 2.5 Workflow Tip: Incremental Development

  When building visualizations:
  1. Start with the most basic chart: `alt.Chart(data).mark_type().encode(x=..., y=...)`
  2. Gradually add complexity: color, size, tooltips, specific binning, etc.
  3. Refine aesthetics and properties: titles, labels, chart width/height.

This makes it easier to understand how each piece contributes and to debug if something goes wrong.

**Example: Building a scatter plot incrementally**

**Step 1**: Basic scatter
```py
chart_step1 = alt.Chart(cars_pl_cleaned).mark_point().encode(
    x='Displacement:Q',
    y='Acceleration:Q'
)
chart_step1.display()
```

**Step 2**: Add color by Origin
```py
chart_step2 = alt.Chart(cars_pl_cleaned).mark_point().encode(
    x='Displacement:Q',
    y='Acceleration:Q',
    color='Origin:N'
)
chart_step2.display()
```

**Step 3**: Add tooltips and a title
```py
 chart_step3 = alt.Chart(cars_pl_cleaned).mark_point().encode(
     x='Displacement:Q',
     y='Acceleration:Q',
     color='Origin:N',
     tooltip=['Name:N', 'Displacement:Q', 'Acceleration:Q']
 ).properties(
     title='Car Displacement vs. Acceleration by Origin',
     width=500
 )
 chart_step3.display()
 ```